In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

BASE = Path("/content/drive/MyDrive/pruebas")

SPLIT_DIR = BASE / "audio_speaker/splits"
OUT_DIR   = BASE / "pre/embeddings/windows_vfinal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = SPLIT_DIR / "aug/train_fin_aug.csv"   # train original + español augmentado
VAL_CSV   = SPLIT_DIR / "val_fin.csv"
TEST_CSV  = SPLIT_DIR / "test_fin.csv"

SR = 16000
WIN_SEC = 2.0
HOP_SEC = 1.0
MAX_WINDOWS_PER_CLIP = 10
MIN_WINDOWS_PER_CLIP = 1

print("TRAIN_CSV:", TRAIN_CSV)
print("VAL_CSV  :", VAL_CSV)
print("TEST_CSV :", TEST_CSV)
print("OUT_DIR  :", OUT_DIR)

TRAIN_CSV: /content/drive/MyDrive/pruebas/audio_speaker/splits/aug/train_fin_aug.csv
VAL_CSV  : /content/drive/MyDrive/pruebas/audio_speaker/splits/val_fin.csv
TEST_CSV : /content/drive/MyDrive/pruebas/audio_speaker/splits/test_fin.csv
OUT_DIR  : /content/drive/MyDrive/pruebas/pre/embeddings/windows_vfinal


In [3]:
df_tr = pd.read_csv(TRAIN_CSV)
df_va = pd.read_csv(VAL_CSV)
df_te = pd.read_csv(TEST_CSV)

print("train:", df_tr.shape)
print("val  :", df_va.shape)
print("test :", df_te.shape)

display(df_tr.head(2))
display(df_va.head(2))
display(df_te.head(2))

train: (11864, 16)
val  : (2945, 12)
test : (3165, 12)


,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final,group_id,path_norm,sr,is_aug,aug_ops
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_1-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,16000,False,NaN
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_16-02-02-04,2.0,2.0,4.0,Audio10,NaN,Audio10,es_Audio10,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,16000,False,NaN


,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final,group_id
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_10-02-03-04,2.0,3.0,4.0,Audio1,NaN,Audio1,es_Audio1
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_100-03-03-04,3.0,3.0,4.0,Audio1,NaN,Audio1,es_Audio1


,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final,group_id
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio6,Audio6_101-04-03-04,4.0,3.0,4.0,Audio6,NaN,Audio6,es_Audio6
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio6,Audio6_103-04-03-04,4.0,3.0,4.0,Audio6,NaN,Audio6,es_Audio6


In [4]:
def prepare_split_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # path_norm: si no existe, usar path
    if "path_norm" not in df.columns:
        df["path_norm"] = df["path"]
    else:
        df["path_norm"] = df["path_norm"].fillna(df["path"])

    # is_aug
    if "is_aug" not in df.columns:
        df["is_aug"] = False
    df["is_aug"] = df["is_aug"].astype("boolean").fillna(False)

    # aug_ops
    if "aug_ops" not in df.columns:
        df["aug_ops"] = ""
    df["aug_ops"] = df["aug_ops"].astype("string").fillna("")

    # lang
    if "lang" not in df.columns:
        df["lang"] = "xx"

    # speaker final, si existe
    if "speaker_id_final" not in df.columns and "speaker_id" in df.columns:
        df["speaker_id_final"] = df["speaker_id"]

    return df.reset_index(drop=True)

In [5]:
df_tr = prepare_split_df(df_tr)
df_va = prepare_split_df(df_va)
df_te = prepare_split_df(df_te)

print(df_tr[["path", "path_norm", "lang", "is_aug", "aug_ops"]].head(3))

                                                path  \
0  /content/drive/.shortcut-targets-by-id/1Kj75Ss...   
1  /content/drive/.shortcut-targets-by-id/1Kj75Ss...   
2  /content/drive/.shortcut-targets-by-id/1Kj75Ss...   

                                           path_norm lang  is_aug aug_ops  
0  /content/drive/.shortcut-targets-by-id/1Kj75Ss...   es   False          
1  /content/drive/.shortcut-targets-by-id/1Kj75Ss...   es   False          
2  /content/drive/.shortcut-targets-by-id/1Kj75Ss...   es   False          


In [6]:
# Funciones de audio y ventaneo

def load_mono_sr16(path, target_sr=SR):
    y, sr = sf.read(str(path), always_2d=False)

    if y.ndim == 2:
        y = y.mean(axis=1)

    y = y.astype(np.float32, copy=False)

    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr, res_type="soxr_vhq")
        sr = target_sr

    return y, sr


def frame_indices(n_samples, win_s, hop_s, sr, ensure_min=True):
    win = int(round(win_s * sr))
    hop = int(round(hop_s * sr))

    if n_samples <= 0:
        return np.array([0], dtype=np.int64), np.array([0], dtype=np.int64)

    if n_samples < win:
        if not ensure_min:
            return np.array([], dtype=np.int64), np.array([], dtype=np.int64)

        center = n_samples // 2
        start = max(0, center - win // 2)
        end = min(n_samples, start + win)
        start = max(0, end - win)

        return np.array([start], dtype=np.int64), np.array([end], dtype=np.int64)

    starts = np.arange(0, n_samples - win + 1, hop, dtype=np.int64)
    ends = starts + win
    return starts, ends


def equispaced_idx(n, k):
    if k >= n:
        return np.arange(n, dtype=np.int64)

    pos = np.linspace(0, n - 1, num=k)
    idx = np.unique(np.round(pos).astype(np.int64))
    return idx


def rms_vec(x):
    return float(np.sqrt(np.mean(x * x) + 1e-12))

In [7]:
# Ventaneo por split

def windows_from_df(df: pd.DataFrame, split_name="train") -> pd.DataFrame:
    rows = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"Ventaneo {split_name}"):
        p = row["path_norm"]

        try:
            y, sr = load_mono_sr16(p)
        except Exception as e:
            print(f"[WARN] No se pudo leer: {p} | {e}")
            continue

        s_arr, e_arr = frame_indices(
            len(y),
            win_s=WIN_SEC,
            hop_s=HOP_SEC,
            sr=sr,
            ensure_min=True
        )

        if len(s_arr) > MAX_WINDOWS_PER_CLIP:
            keep = equispaced_idx(len(s_arr), MAX_WINDOWS_PER_CLIP)
            s_arr, e_arr = s_arr[keep], e_arr[keep]

        if len(s_arr) < MIN_WINDOWS_PER_CLIP:
            s_arr = np.array([0], dtype=np.int64)
            e_arr = np.array([min(len(y), int(WIN_SEC * sr))], dtype=np.int64)

        for s, e in zip(s_arr, e_arr):
            seg = y[s:e]

            rows.append({
                "clip_idx": i,
                "path": p,
                "orig_path": row.get("path", p),
                "clip_id": row.get("clip_id", ""),
                "speaker_id_final": row.get("speaker_id_final", ""),
                "lang": row.get("lang", "xx"),
                "is_aug": bool(row.get("is_aug", False)),
                "aug_ops": str(row.get("aug_ops", "")),
                "valence": float(row["valence"]),
                "arousal": float(row["arousal"]),
                "dominance": float(row["dominance"]),
                "start_samp": int(s),
                "end_samp": int(e),
                "win_rms": rms_vec(seg),
            })

    return pd.DataFrame(rows)

Generar ventanas

In [8]:
dfw_te = windows_from_df(df_te, "test")
print("Ventanas test :", dfw_te.shape)

Ventaneo test:   0%|          | 0/3165 [00:00<?, ?it/s]

Ventanas test : (9980, 14)


In [9]:
dfw_te.to_csv(OUT_DIR / "windows_test_raw.csv", index=False)

In [17]:
dfw_te = pd.read_csv(OUT_DIR / "windows_test_raw.csv")

In [18]:
dfw_te.head(2)

,clip_idx,path,orig_path,clip_id,speaker_id_final,lang,is_aug,aug_ops,valence,arousal,dominance,start_samp,end_samp,win_rms
0,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio6_101-04-03-04,Audio6,es,False,NaN,4.0,3.0,4.0,0,32000,0.270005
1,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio6_101-04-03-04,Audio6,es,False,NaN,4.0,3.0,4.0,16000,48000,0.259570


In [8]:
dfw_va = windows_from_df(df_va, "val")
print("Ventanas val  :", dfw_va.shape)

Ventaneo val:   0%|          | 0/2945 [00:00<?, ?it/s]

Ventanas val  : (8459, 14)


In [9]:
dfw_va.to_csv(OUT_DIR / "windows_val_raw.csv", index=False)

In [15]:
dfw_va.head(2)

,clip_idx,path,orig_path,clip_id,speaker_id_final,lang,is_aug,aug_ops,valence,arousal,dominance,start_samp,end_samp,win_rms
0,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio1_10-02-03-04,Audio1,es,False,,2.0,3.0,4.0,0,32000,0.192927
1,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio1_10-02-03-04,Audio1,es,False,,2.0,3.0,4.0,16000,48000,0.208613


In [11]:
df_te.shape

(3165, 15)

In [10]:
df_tr.shape

(11864, 16)

In [12]:
dfw_tr = windows_from_df(df_tr, "train")
print("Ventanas train:", dfw_tr.shape)

Ventaneo train:   0%|          | 0/11864 [00:00<?, ?it/s]

Ventanas train: (38249, 14)


In [13]:
dfw_tr.to_csv(OUT_DIR / "windows_train_raw.csv", index=False)

In [14]:
display(dfw_tr.head(3))

,clip_idx,path,orig_path,clip_id,speaker_id_final,lang,is_aug,aug_ops,valence,arousal,dominance,start_samp,end_samp,win_rms
0,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio10_1-02-02-04,Audio10,es,False,,2.0,2.0,4.0,0,32000,0.242901
1,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio10_1-02-02-04,Audio10,es,False,,2.0,2.0,4.0,16000,48000,0.201528
2,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio10_1-02-02-04,Audio10,es,False,,2.0,2.0,4.0,32000,64000,0.157499


Log-scaling of RMS compresses inter-speaker energy differences, enabling the model to focus on relative emotional intensity rather than absolute signal amplitude.

In [20]:
def recompute_rms_grouped(df):
    df = df.copy()
    rms_out = np.zeros(len(df), dtype=np.float32)

    for path, g in tqdm(df.groupby("path", sort=False), total=df["path"].nunique()):
        y, sr = load_mono_sr16(path)

        for idx, row in g.iterrows():
            seg = y[int(row["start_samp"]):int(row["end_samp"])]
            rms_out[idx] = rms_vec(seg)

    df["win_rms"] = rms_out
    return df

In [22]:
dfw_tr = recompute_rms_grouped(dfw_tr)

  0%|          | 0/11864 [00:00<?, ?it/s]

In [23]:
dfw_va = recompute_rms_grouped(dfw_va)

  0%|          | 0/2945 [00:00<?, ?it/s]

In [24]:
dfw_te = recompute_rms_grouped(dfw_te)

  0%|          | 0/3165 [00:00<?, ?it/s]

In [25]:
mu_rms = dfw_tr["win_rms"].mean()
sd_rms = dfw_tr["win_rms"].std(ddof=1) + 1e-9

print("mu_rms:", mu_rms)
print("sd_rms:", sd_rms)

mu_rms: 0.09764734
sd_rms: 0.09354862670762634


In [26]:
eps = 1e-6
log_std = np.std(np.log1p(dfw_tr["win_rms"] + eps), ddof=1) + 1e-9
log_mu = np.log1p(mu_rms + eps)

for d in [dfw_tr, dfw_va, dfw_te]:
    d["zrms_log"] = (np.log1p(d["win_rms"] + eps) - log_mu) / log_std

In [28]:
dfw_tr

,clip_idx,path,orig_path,clip_id,speaker_id_final,lang,is_aug,aug_ops,valence,arousal,dominance,start_samp,end_samp,win_rms,zrms_log
0,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio10_1-02-02-04,Audio10,es,False,,2.0,2.0,4.0,0,32000,0.242901,1.547708
1,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio10_1-02-02-04,Audio10,es,False,,2.0,2.0,4.0,16000,48000,0.201528,1.126111
2,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio10_1-02-02-04,Audio10,es,False,,2.0,2.0,4.0,32000,64000,0.157499,0.661191
3,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio10_1-02-02-04,Audio10,es,False,,2.0,2.0,4.0,48000,80000,0.162952,0.719716
4,0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio10_1-02-02-04,Audio10,es,False,,2.0,2.0,4.0,64000,96000,0.169380,0.788365
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38244,11862,/content/drive/MyDrive/pruebas/pre/train_aug/e...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio13_66-03-03-03,Audio13,es,True,rvb0.14s_0.10,3.0,3.0,3.0,0,32000,0.049239,-0.561703
38245,11862,/content/drive/MyDrive/pruebas/pre/train_aug/e...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio13_66-03-03-03,Audio13,es,True,rvb0.14s_0.10,3.0,3.0,3.0,16000,48000,0.034994,-0.731934
38246,11862,/content/drive/MyDrive/pruebas/pre/train_aug/e...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio13_66-03-03-03,Audio13,es,True,rvb0.14s_0.10,3.0,3.0,3.0,32000,64000,0.046931,-0.589127
38247,11862,/content/drive/MyDrive/pruebas/pre/train_aug/e...,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,Audio13_66-03-03-03,Audio13,es,True,rvb0.14s_0.10,3.0,3.0,3.0,48000,80000,0.058710,-0.449801


In [29]:
def to_npz(dfw, out_path):
    np.savez_compressed(
        out_path,
        path=np.array(dfw["path"].astype(str).tolist()),
        orig_path=np.array(dfw["orig_path"].astype(str).tolist()) if "orig_path" in dfw.columns else np.array(dfw["path"].astype(str).tolist()),
        clip_id=np.array(dfw["clip_id"].astype(str).tolist()) if "clip_id" in dfw.columns else np.array([""] * len(dfw)),
        speaker_id_final=np.array(dfw["speaker_id_final"].astype(str).tolist()) if "speaker_id_final" in dfw.columns else np.array([""] * len(dfw)),
        clip_idx=dfw["clip_idx"].to_numpy(np.int64),
        start_samp=dfw["start_samp"].to_numpy(np.int64),
        end_samp=dfw["end_samp"].to_numpy(np.int64),
        valence=dfw["valence"].to_numpy(np.float32),
        arousal=dfw["arousal"].to_numpy(np.float32),
        dominance=dfw["dominance"].to_numpy(np.float32),
        win_rms=dfw["win_rms"].to_numpy(np.float32),
        zrms_log=dfw["zrms_log"].to_numpy(np.float32),
        lang=np.array(dfw["lang"].astype(str).tolist()),
        is_aug=dfw["is_aug"].to_numpy(bool) if "is_aug" in dfw.columns else np.zeros(len(dfw), dtype=bool),
        aug_ops=np.array(dfw["aug_ops"].astype(str).tolist()) if "aug_ops" in dfw.columns else np.array([""] * len(dfw)),
        sr=np.int64(16000),
        win_sec=np.float32(2.0),
        hop_sec=np.float32(1.0),
        max_win_per_clip=np.int64(10),
        min_win_per_clip=np.int64(1),
        mu_rms=np.float32(mu_rms),
        sd_rms=np.float32(sd_rms),
    )

In [30]:
to_npz(dfw_tr, OUT_DIR / "windows_train.npz")

In [31]:
to_npz(dfw_va, OUT_DIR / "windows_val.npz")
to_npz(dfw_te, OUT_DIR / "windows_test.npz")

In [32]:
mu_rms

np.float32(0.09764734)